Recomandation Systems

ML systems used to get what a user would like.

Types of Recomendation System

1.) Content Based  - recomends based on the similarity of content.

2.) Collaborative Filltering - recomends based on the users interest.
eg- if A and B have similar likes, so if what A likes it will also be shown to B.

3.) Hybrid - Combination of both content and collaborative.

Project Flow

Data -> Preprocessing of Data -> Model Building ->  Webisite Building -> Deployment

Dataset: https://www.kaggle.com/datasets/amritvirsinghx/web-series-ultimate-edition

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer
import pickle

1.) Data Preprocessing

In [2]:
shows = pd.read_csv('All_Streaming_Shows.csv')

In [3]:
shows.head()

,Series Title,Year Released,Content Rating,IMDB Rating,R Rating,Genre,Description,No of Seasons,Streaming Platform
0,Breaking Bad,2008,18+,9.5,100,"Crime,Drama","When Walter White, a New Mexico chemistry teac...",5Seasons,Netflix
1,Game of Thrones,2011,18+,9.3,99,"Action & Adventure,Drama",Seven noble families fight for control of the ...,8Seasons,"HBO MAX,HBO"
2,Rick and Morty,2013,18+,9.2,97,"Animation,Comedy",Rick is a mentally-unbalanced but scientifical...,4Seasons,"Free Services,HBO MAX,Hulu"
3,Stranger Things,2016,16+,8.8,96,"Drama,Fantasy","When a young boy vanishes, a small town uncove...",3Seasons,Netflix
4,The Boys,2019,18+,8.7,95,"Action & Adventure,Comedy",A group of vigilantes known informally as “The...,2Seasons,Prime Video


In [4]:
shows.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12353 entries, 0 to 12352
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Series Title        12353 non-null  object 
 1   Year Released       12353 non-null  int64  
 2   Content Rating      7232 non-null   object 
 3   IMDB Rating         10207 non-null  float64
 4   R Rating            12353 non-null  int64  
 5   Genre               12353 non-null  object 
 6   Description         12353 non-null  object 
 7   No of Seasons       12353 non-null  object 
 8   Streaming Platform  10370 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 868.7+ KB


1.1) Selecting Useful Columns

Focus on the descriptive fields needed for a content-based recommender.

In [5]:
shows = shows[['Series Title', 'Genre', 'Description', 'Streaming Platform', 'Content Rating', 'Year Released']].copy()

In [6]:
shows.head(1)

,Series Title,Genre,Description,Streaming Platform,Content Rating,Year Released
0,Breaking Bad,"Crime,Drama","When Walter White, a New Mexico chemistry teac...",Netflix,18+,2008


1.2) Renaming Columns

Rename to simpler field names that mirror our downstream processing.

Columns renamed:
- Series Title → title
- Genre → genres
- Description → description
- Streaming Platform → platforms
- Content Rating → content_rating
- Year Released → year

In [7]:
shows.rename(columns={'Series Title': 'title',
                                  "Genre": 'genres',
                                  "Description": 'description',
                                  "Streaming Platform": 'platforms',
                                  "Content Rating": 'content_rating',
                                  "Year Released": 'year'},
                inplace=True)

In [8]:
shows.head()

,title,genres,description,platforms,content_rating,year
0,Breaking Bad,"Crime,Drama","When Walter White, a New Mexico chemistry teac...",Netflix,18+,2008
1,Game of Thrones,"Action & Adventure,Drama",Seven noble families fight for control of the ...,"HBO MAX,HBO",18+,2011
2,Rick and Morty,"Animation,Comedy",Rick is a mentally-unbalanced but scientifical...,"Free Services,HBO MAX,Hulu",18+,2013
3,Stranger Things,"Drama,Fantasy","When a young boy vanishes, a small town uncove...",Netflix,16+,2016
4,The Boys,"Action & Adventure,Comedy",A group of vigilantes known informally as “The...,Prime Video,18+,2019


1.3) Missing Data

Drop rows missing critical descriptive information.

In [9]:
shows.isnull().sum()

title                0
genres               0
description          0
platforms         1983
content_rating    5121
year                 0
dtype: int64

In [10]:
shows.dropna(subset=['title', 'genres', 'description'], inplace=True)
shows.reset_index(drop=True, inplace=True)

1.4) Duplicate Data

Ensure each series title appears only once.

In [11]:
shows.duplicated(subset='title').sum()

np.int64(244)

In [12]:
shows.drop_duplicates(subset='title', inplace=True)
shows.reset_index(drop=True, inplace=True)

1.5) Feature Cleaning

Parse multi-value fields and normalize tokens.

In [13]:
shows.iloc[0].genres

'Crime,Drama'

Clean genre, platform, and metadata columns

In [14]:
def split_and_strip(value):
    if pd.isna(value):
        return []
    tokens = []
    for token in str(value).split(','):
        cleaned = token.strip()
        if not cleaned or cleaned.lower() == 'nan':
            continue
        tokens.append(cleaned.replace(' ', ''))
    return tokens

def tokenize_description(text):
    if pd.isna(text):
        return []
    return str(text).split()

def normalize_single_token(value):
    if pd.isna(value):
        return []
    cleaned = str(value).strip()
    if not cleaned or cleaned.lower() == 'nan':
        return []
    return [cleaned.replace(' ', '')]

In [15]:
shows['genres'] = shows['genres'].apply(split_and_strip)

In [16]:
shows['platforms'] = shows['platforms'].apply(split_and_strip)

In [ ]:
shows[['title', 'genres', 'platforms']].head()

,genres,id,keywords,original_title,overview,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,"[Action, Adventure, Crime]",206647,"[spy, based on novel, secret agent, sequel, mi...",Spectre,A cryptic message from Bond’s past sends him o...,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,"[Action, Crime, Drama, Thriller]",49026,"[dc comics, crime fighter, terrorist, secret i...",The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,"[Action, Adventure, Science Fiction]",49529,"[based on novel, mars, medallion, space travel...",John Carter,"John Carter is a war-weary, former military ca...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


Tokenize Description Text

In [17]:
shows['description'] = shows['description'].apply(tokenize_description)

In [18]:
shows['content_rating'] = shows['content_rating'].apply(normalize_single_token)

In [19]:
shows[['title', 'content_rating']].head()

,title,content_rating
0,Breaking Bad,[18+]
1,Game of Thrones,[18+]
2,Rick and Morty,[18+]
3,Stranger Things,[16+]
4,The Boys,[18+]


Normalize Release Year

In [20]:
shows['year'] = shows['year'].apply(normalize_single_token)

In [21]:
shows[['title', 'year']].head()

,title,year
0,Breaking Bad,[2008]
1,Game of Thrones,[2011]
2,Rick and Morty,[2013]
3,Stranger Things,[2016]
4,The Boys,[2019]


In [22]:
shows.sample(3)[['title', 'platforms', 'genres']]

,title,platforms,genres
4890,The Piano Forest,[Netflix],"[Anime, Animation]"
9947,CBS Sunday Morning,"[FreeServices, CBSAllAccess]","[2016, FreeServices]"
9204,Mud Lovin' Rednecks,[],"[Reality, 2011]"


Assemble Token Lists

Combine text tokens with categorical metadata.

In [23]:
def assemble_tags(row):
    tokens = []
    for column in ['description', 'genres', 'platforms', 'content_rating', 'year']:
        tokens.extend(row[column])
    return tokens

shows['tags'] = shows.apply(assemble_tags, axis=1)
shows[['title', 'tags']].head()

,title,tags
0,Breaking Bad,"[When, Walter, White,, a, New, Mexico, chemist..."
1,Game of Thrones,"[Seven, noble, families, fight, for, control, ..."
2,Rick and Morty,"[Rick, is, a, mentally-unbalanced, but, scient..."
3,Stranger Things,"[When, a, young, boy, vanishes,, a, small, tow..."
4,The Boys,"[A, group, of, vigilantes, known, informally, ..."


In [24]:
shows.sample(5)[['title', 'tags']]

,title,tags
863,Frank Herbert's Children of Dune,"[Frank, Herbert's, Children, of, Dune, is, a, ..."
1998,River Monsters,"[Extreme, angler, Jeremy, Wade, is, on, the, h..."
11120,"Sesame Street: 3,2,1 Let's Go","[Sesame, Street:, 3,2,1, Let's, Go, has, one, ..."
7668,Cheerleader Nation,"[Cheerleader, Nation, is, a, US, television, s..."
10749,Pinkfong! Healthy Habit Songs,"[Pinkfong!, Healthy, Habit, Songs, has, one, o..."


Normalize Tokens

Lowercase and join tag tokens for vectorization.

In [25]:
shows['tags'] = shows['tags'].apply(lambda x: [token.lower() for token in x])

In [26]:
shows['tags'] = shows['tags'].apply(lambda x: " ".join(x))

1.6) Final Tag Corpus

All descriptive signals now live in `tags`, which we use for similarity modelling.

In [27]:
shows[['title', 'tags']].head()

,title,tags
0,Breaking Bad,"when walter white, a new mexico chemistry teac..."
1,Game of Thrones,seven noble families fight for control of the ...
2,Rick and Morty,rick is a mentally-unbalanced but scientifical...
3,Stranger Things,"when a young boy vanishes, a small town uncove..."
4,The Boys,a group of vigilantes known informally as “the...


In [28]:
shows.sample(3)[['title', 'tags']]

,title,tags
6988,Daisy of Love,daisy of love is an american reality televisio...
10115,Growing Up Supermodel,a group of young models risk it all to live up...
5829,La Patrona,gabriela suárez (aracely arámbula) is the only...


In [29]:
new_df = shows[['title', 'tags']].copy()

In [30]:
new_df.head()

,title,tags
0,Breaking Bad,"when walter white, a new mexico chemistry teac..."
1,Game of Thrones,seven noble families fight for control of the ...
2,Rick and Morty,rick is a mentally-unbalanced but scientifical...
3,Stranger Things,"when a young boy vanishes, a small town uncove..."
4,The Boys,a group of vigilantes known informally as “the...


In [31]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [32]:
new_df.head()

,title,tags
0,Breaking Bad,"when walter white, a new mexico chemistry teac..."
1,Game of Thrones,seven noble families fight for control of the ...
2,Rick and Morty,rick is a mentally-unbalanced but scientifical...
3,Stranger Things,"when a young boy vanishes, a small town uncove..."
4,The Boys,a group of vigilantes known informally as “the...


1.7) Stemming

Reduce vocabulary size by converting words to their root form.

In [33]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x.split()))

In [34]:
new_df.head()

,title,tags
0,Breaking Bad,"when walter white, a new mexico chemistry teac..."
1,Game of Thrones,seven noble families fight for control of the ...
2,Rick and Morty,rick is a mentally-unbalanced but scientifical...
3,Stranger Things,"when a young boy vanishes, a small town uncove..."
4,The Boys,a group of vigilantes known informally as “the...


In [35]:
ps = PorterStemmer()

def stem(text):
    return " ".join(ps.stem(word) for word in text.split())

new_df['tags'] = new_df['tags'].apply(stem)

2.) Vectorization

Transform the processed tag text into numerical features for similarity search.

Bag of Words

Convert each show's combined tags into a vector of word counts (excluding English stop words).

In [36]:
cv = CountVectorizer(max_features =5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [37]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(12109, 5000))

In [39]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

Compute Cosine Similarity

Measure pairwise similarity between show vectors.

In [38]:
similarity = cosine_similarity(vectors)
similarity.shape

(12109, 12109)

3.) Recommendation Function

Return the most similar shows for a given title.

In [39]:
def recommend(show_title):
    if show_title not in new_df['title'].values:
        print(f"Show '{show_title}' not found in the catalog.")
        return
    
    show_index = new_df[new_df['title'] == show_title].index[0]
    distances = similarity[show_index]
    shows_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"\nRecommendations for '{show_title}':\n")
    for rank, (idx, score) in enumerate(shows_list, 1):
        print(f"{rank}. {new_df.iloc[idx]['title']} (similarity: {score:.3f})")

In [44]:
recommend("Marvel's Jessica Jones")


Recommendations for 'Marvel's Jessica Jones':

1. Marvel's The Defenders (similarity: 0.714)
2. Marvel's Iron Fist (similarity: 0.685)
3. Marvel's Daredevil (similarity: 0.683)
4. Marvel's Luke Cage (similarity: 0.680)
5. Kojak (similarity: 0.674)


4.) Export Artifacts

Persist the processed data and similarity matrix for deployment.

In [41]:
pickle.dump(new_df, open('shows.pkl', 'wb'))

In [42]:
pickle.dump(similarity, open('shows_similarity.pkl', 'wb'))